In [4]:
from pathlib import Path

import pandas as pd


SILVER_FILE = Path(
    "../data/processed/support_tickets_clean.csv"
)

df = pd.read_csv(
    SILVER_FILE,
    parse_dates=[
        "date_of_purchase",
        "first_response_at",
        "resolution_at",
    ],
)

print("Shape:", df.shape)
print("Unique ticket IDs:", df["ticket_id"].nunique())
print("Duplicate ticket IDs:", df["ticket_id"].duplicated().sum())

Shape: (8469, 29)
Unique ticket IDs: 8469
Duplicate ticket IDs: 0


In [5]:
dimension_candidates = {
    "customer": ["customer_key"], 
    "product": ["product_purchased"], 
    "issue": ["ticket_type", "ticket_subject"], 
    "channel": ["ticket_channel"],
    "priority": ["ticket_priority"],
    "status": ["ticket_status"],
    "purchase_date": ["date_of_purchase"],
}

for dimension_name, columns in dimension_candidates.items():
    member_count = (
        df[columns]
        .drop_duplicates()
        .shape[0]
    )

    print(
        f"{dimension_name}: "
        f"{member_count:,} members"
    )

customer: 8,320 members
product: 42 members
issue: 80 members
channel: 4 members
priority: 4 members
status: 3 members
purchase_date: 730 members


In [6]:
customer_profile = (
    df.groupby("customer_key")
    .agg(
        ticket_count=(
            "ticket_id", 
            "count",
        ), 
        age_versions=(
            "customer_age", 
            "nunique",
        ),
        gender_versions=(
            "customer_gender",
            "nunique",
        ), 
        name_versions=(
            "customer_name", 
            "nunique",
        ),
    )
    .reset_index()
)

customer_conflicts = customer_profile[
    (customer_profile["age_versions"] > 1) |
    (customer_profile["gender_versions"] > 1) |
    (customer_profile["name_versions"] > 1)
]

print(
    "Customers with conflicting attributes:",
    len(customer_conflicts),
)

display(customer_conflicts.head(10))

Customers with conflicting attributes: 139


,customer_key,ticket_count,age_versions,gender_versions,name_versions
16,00af78366eeb21b2,2,2,2,2
23,00d5780df63ddebe,2,2,2,2
61,01eee67dc5b01a91,2,2,1,2
70,0276696572f4a9e6,2,2,2,2
103,0394e2676f522ce6,2,2,1,2
163,05658114e0c8a4f9,2,2,2,2
205,06c0455bc3b80e64,2,2,2,2
229,076dfa0718f86c17,2,2,2,2
235,079ae991f11769c6,2,2,2,2
287,09302a1dd47275fd,2,2,2,2


In [7]:
dimension_key_columns = [
    "customer_key",
    "product_purchased",
    "ticket_type",
    "ticket_subject",
    "ticket_channel",
    "ticket_priority",
    "ticket_status",
]

print(
    df[dimension_key_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

customer_key         0
product_purchased    0
ticket_type          0
ticket_subject       0
ticket_channel       0
ticket_priority      0
ticket_status        0
dtype: int64


In [8]:
issue_profile = (
    df.groupby(
        [
            "ticket_type",
            "ticket_subject",
        ], 
        dropna=False,
    )
    .agg(
        ticket_count=(
            "ticket_id", 
            "count",
        )
    )
    .reset_index()
    .sort_values(
        "ticket_count", 
        ascending=False,
    )
)

print(
    "Unique issue combinations:", 
    len(issue_profile)
)

display(issue_profile.head(20))

Unique issue combinations: 80


,ticket_type,ticket_subject,ticket_count
54,Refund Request,Hardware issue,129
30,Cancellation Request,Refund request,126
11,Billing Inquiry,Product compatibility,123
15,Billing Inquiry,Software bug,122
46,Product Inquiry,Refund request,122
72,Technical Issue,Network problem,122
43,Product Inquiry,Product compatibility,121
55,Refund Request,Installation support,119
75,Technical Issue,Product compatibility,119
49,Refund Request,Battery life,119


In [9]:
print(
    "Minimum purchase date:",
    df["date_of_purchase"].min(),
)

print(
    "Maximum purchase date:",
    df["date_of_purchase"].max(),
)

print(
    "Unique purchase dates:",
    df["date_of_purchase"].nunique(),
)

Minimum purchase date: 2020-01-01 00:00:00
Maximum purchase date: 2021-12-30 00:00:00
Unique purchase dates: 730


In [10]:
purchase_date_coverage = (
    df["date_of_purchase"]
    .dt.normalize()
    .sort_values()
    .drop_duplicates()
)

expected_calendar = pd.date_range(
    start=df["date_of_purchase"].min(),
    end=df["date_of_purchase"].max(),
    freq="D",
)

print(
    "Calendar days in full range:", 
    len(expected_calendar),
)

print(
    "Purchase dates present:", 
    len(purchase_date_coverage)
)

print(
    "Missing calendar dates:", 
    len(expected_calendar)
    - len(purchase_date_coverage),
)

Calendar days in full range: 730
Purchase dates present: 730
Missing calendar dates: 0
